# Глава 6. Первые модели машинного обучения — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Что такое машинное обучение

До сих пор правила придумывали мы: «если курьер пеший — скорее опоздает». Машинное обучение переворачивает задачу: мы даём примеры с ответами, а модель сама находит правила. Знакомимся с языком ML — признаки, целевой признак, обучение, предсказание, — учимся переводить запрос заказчика в ML-задачу нужного типа и обучаем первую модель в scikit-learn.

Теория — на сайте курса: `ch06/01-what-is-ml.html`. Задачи «собери код из строк» (1) остались на сайте.

### Признаки и цель · разминка

Соберите `X` — таблицу из признаков `distance_km`, `items`, `promised_min` (в этом порядке) и `y` — Series с опозданием (`actual_min > promised_min`). Доставки — с временем 1–299 минут.

*Подсказки (2): `dsc.hint('xy')`*

In [ ]:
%%task ch06/01-what-is-ml/xy
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]

### Базовая модель · разминка

Сохраните в `base_acc` долю правильных ответов модели «всегда вовремя» (3 знака).

*Подсказки (2): `dsc.hint('ch06/01-what-is-ml/baseline')`*

In [ ]:
%%task ch06/01-what-is-ml/baseline
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
y = ok["actual_min"] > ok["promised_min"]

### Какой это тип задачи · разминка

Определите тип задачи для каждого запроса. Заполните словарь `kinds`: ключ — номер запроса, значение — одна из строк `"классификация"`, `"регрессия"`, `"кластеризация"`.

1. «Уйдёт ли гость в ближайшие 90 дней?»
2. «Сколько выручки сделает кофейня на Баумана завтра?»
3. «Разбейте гостей на группы по привычкам — сколько групп, не знаем».
4. «Какой из пяти напитков гость закажет следующим?»
5. «Через сколько минут приедет курьер?»

*Подсказки (2): `dsc.hint('task-kind')`*

In [ ]:
%%task ch06/01-what-is-ml/task-kind
kinds = {
    1: "...",
    2: "...",
    3: "...",
    4: "...",
    5: "...",
}

### Правило аналитика · база

Сравните два ручных правила: «опоздает, если курьер пеший» и «опоздает, если обещали 30 минут». Сохраните `acc_walk` и `acc_30` — доли правильных ответов каждого правила (3 знака), и `better` — название лучшего правила: `"пеший"` или `"30 минут"`.

*Подсказки (2): `dsc.hint('ch06/01-what-is-ml/rule')`*

In [ ]:
%%task ch06/01-what-is-ml/rule
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
y = ok["actual_min"] > ok["promised_min"]

### Первое дерево · база

Обучите `DecisionTreeClassifier(max_depth=3, random_state=0)` на признаках `distance_km`, `items`, `promised_min`, `hour` (час из `created`). Сохраните модель в `model` и долю правильных ответов на тех же данных в `train_acc` (3 знака).

*Подсказки (2): `dsc.hint('first-tree')`*

In [ ]:
%%task ch06/01-what-is-ml/first-tree
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour

### Предсказание для новых заказов · база

Обучите то же дерево и предскажите опоздание для трёх новых заказов. Сохраните **список** ответов `True/False` в `preds`. `predict` возвращает массив numpy, и его элементы печатаются как `np.True_` / `np.False_`; обычные `True/False` получаются через `bool(x)`.

| distance_km | items | promised_min | hour |
|---|---|---|---|
| 1.0 | 1 | 30 | 18 |
| 1.2 | 4 | 30 | 13 |
| 11.0 | 2 | 60 | 19 |

*Подсказки (2): `dsc.hint('ch06/01-what-is-ml/new-orders')`*

In [ ]:
%%task ch06/01-what-is-ml/new-orders
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
cols = ["distance_km", "items", "promised_min", "hour"]
model = DecisionTreeClassifier(max_depth=3, random_state=0).fit(ok[cols], ok["actual_min"] > ok["promised_min"])

### Две постановки одного запроса · база

Запрос логиста: «В момент передачи заказа курьеру хочу знать, опоздает ли он и на сколько минут». Для доставок с временем 1–299 минут сохраните:

- `leaks` — список столбцов `deliveries.csv`, которые становятся известны только **после** доставки (отсортированный по алфавиту);
- `y_cls` — целевой признак классификации: опоздает ли (True/False);
- `y_reg` — целевой признак регрессии: на сколько минут фактическое время больше обещанного (отрицательное — приехали раньше);
- `late_share` — долю опозданий (3 знака) и `late_median` — медиану опоздания в минутах **среди опоздавших** доставок.

*Подсказки (3): `dsc.hint('frame')`*

In [ ]:
%%task ch06/01-what-is-ml/frame
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
print(d.columns.tolist())

### Утечка · посложнее

Обучите дерево глубины 3 дважды: на признаках `distance_km`, `items`, `promised_min` (`acc_fair`) и на них же плюс `actual_min` (`acc_leak`). Обе доли — 3 знака. В `gap` — разница (п.п., 1 знак).

*Подсказки (2): `dsc.hint('ch06/01-what-is-ml/leak')`*

In [ ]:
%%task ch06/01-what-is-ml/leak
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
y = ok["actual_min"] > ok["promised_min"]

### Регрессия: базовая модель · посложнее

Для регрессии (предсказать `actual_min` — минуты) самая простая модель — всегда отвечать средним. Сохраните `mae_mean` — среднюю абсолютную ошибку такой модели (среднее модулей «факт − прогноз», 2 знака) и `mae_median` — то же, если отвечать медианой.

*Подсказки (2): `dsc.hint('mae-base')`*

In [ ]:
%%task ch06/01-what-is-ml/mae-base
import pandas as pd
d = pd.read_csv("deliveries.csv")
t = d.loc[d["actual_min"].between(1, 299), "actual_min"]

### Первый вопрос дерева · со звёздочкой

Обучите `DecisionTreeClassifier(max_depth=1, random_state=0)` на признаках `distance_km`, `items`, `promised_min`, `hour`. Дерево глубины 1 задаёт ровно один вопрос. Сохраните `feature` — название признака в этом вопросе, `threshold` — порог (2 знака) и `acc` — долю правильных ответов (3 знака).

*Подсказки (2): `dsc.hint('first-split')` · задача по желанию*

In [ ]:
%%task ch06/01-what-is-ml/first-split
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
cols = ["distance_km", "items", "promised_min", "hour"]
y = ok["actual_min"] > ok["promised_min"]

## Урок 2. Обучающая и тестовая выборки

В прошлом уроке мы проверяли модель на тех же доставках, на которых она училась, — это как давать на экзамене билеты, которые ученик уже видел. Учимся проверять честно: откладываем часть данных, на которые модель не смотрит, и разбираемся, что такое переобучение.

Теория — на сайте курса: `ch06/02-split.html`. Задачи «собери код из строк» (1) остались на сайте.

### Разбить данные · разминка

Разбейте `X` и `y` на обучающую и тестовую части: в тесте 25 %, `random_state=42`. Сохраните четыре переменные `X_train`, `X_test`, `y_train`, `y_test` и выведите размер обучающей и тестовой выборок (`len`) через пробел.

*Подсказки (2): `dsc.hint('ch06/02-split/split')`*

In [ ]:
%%task ch06/02-split/split
import pandas as pd
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
X = ok[["distance_km", "items", "promised_min"]]
y = ok["actual_min"] > ok["promised_min"]

### Базовая модель на тесте · разминка

Модели сравнивают на одном и том же тесте. Разбейте данные (25 %, `random_state=42`) и сохраните в `base_test` долю правильных ответов модели «всегда вовремя» **на тестовой части** (3 знака).

*Подсказки (2): `dsc.hint('base-test')`*

In [ ]:
%%task ch06/02-split/base-test
import pandas as pd
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
X = ok[["distance_km", "items", "promised_min"]]
y = ok["actual_min"] > ok["promised_min"]

### Разрыв глубокого дерева · база

Обучите дерево **без ограничения глубины** (`DecisionTreeClassifier(random_state=0)`) на обучающей части признаков `distance_km`, `items`, `promised_min`, `hour`. Сохраните `train_acc`, `test_acc` (3 знака) и `gap` — разрыв в процентных пунктах (`(train_acc − test_acc) · 100`, 1 знак). В `overfit` запишите `True`, если тестовая доля ниже доли базовой модели «всегда вовремя» на тесте, иначе `False`.

*Подсказки (2): `dsc.hint('deep-gap')`*

In [ ]:
%%task ch06/02-split/deep-gap
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = ok[["distance_km", "items", "promised_min", "hour"]]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Подобрать глубину · база

Для глубин от 1 до 10 обучите деревья и соберите списки `train_scores` и `test_scores` — доли правильных ответов на обучении и тесте (без округления, по порядку глубин). В `best_depth` сохраните глубину с наибольшей долей на тесте (если таких несколько — самую маленькую).

*Подсказки (2): `dsc.hint('depth-curve')`*

In [ ]:
%%task ch06/02-split/depth-curve
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = ok[["distance_km", "items", "promised_min", "hour"]]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
train_scores, test_scores = [], []

### График переобучения · база

Нарисуйте кривую «глубина → качество» для глубин 1–12: две линии `ax.plot` — сначала обучение, потом тест, с подписями `label` `"обучение"` и `"тест"`, легенду, подпись оси X `"глубина"` и любой заголовок.

*Подсказки (2): `dsc.hint('curve-plot')`*

In [ ]:
%%task ch06/02-split/curve-plot
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = ok[["distance_km", "items", "promised_min", "hour"]]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Стратификация · база

Разбейте данные дважды (25 %, `random_state=42`): без стратификации и со `stratify=y`. Сохраните доли опозданий в тестовой части: `late_plain` и `late_strat` (3 знака), а в `late_all` — долю опозданий во всех данных (3 знака).

*Подсказки (2): `dsc.hint('strat')`*

In [ ]:
%%task ch06/02-split/strat
import pandas as pd
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
X = ok[["distance_km", "items", "promised_min"]]
y = ok["actual_min"] > ok["promised_min"]

### Разброс оценки · посложнее

Проверьте, насколько оценка зависит от разбиения. Для `random_state` разбиения от 0 до 29 обучите дерево глубины 3 и соберите список `scores` долей правильных ответов на тесте. Сохраните `low`, `high` — минимум и максимум, `spread` — разницу `high − low` в п.п. (1 знак). `low` и `high` — 3 знака; `spread` считайте из уже округлённых `low` и `high`.

*Подсказки (2): `dsc.hint('seeds')`*

In [ ]:
%%task ch06/02-split/seeds
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
X = ok[["distance_km", "items", "promised_min", "hour"]]
y = ok["actual_min"] > ok["promised_min"]

### Правило против моделей на тесте · посложнее

Сравните на одном тесте (25 %, `random_state=42`) три подхода: правило «пеший — опоздает», дерево глубины 3 и дерево без ограничения глубины (оба — на признаках `distance_km`, `items`, `promised_min`, `hour`). Сохраните словарь `results` с ключами `"правило"`, `"дерево 3"`, `"дерево без ограничений"` и долями правильных ответов на тесте (3 знака), а в `winner` — ключ с наибольшей долей.

*Подсказки (3): `dsc.hint('rule-test')`*

In [ ]:
%%task ch06/02-split/rule-test
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
cols = ["distance_km", "items", "promised_min", "hour"]

### Потолок для любой модели · со звёздочкой

Почему даже глубокое дерево не набирает 100 % на обучении? Найдите в обучающей части «близнецов» — доставки с одинаковыми значениями всех четырёх признаков. Для каждой группы близнецов лучшее, что может сделать любая модель, — ответить её большинством. Посчитайте `ceiling` — долю обучающих доставок, которые угадает такой «идеальный запоминатель» (3 знака), и `deep_train` — долю правильных ответов на обучении у дерева без ограничения глубины (3 знака).

*Подсказки (2): `dsc.hint('twins')` · задача по желанию*

In [ ]:
%%task ch06/02-split/twins
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
ok["late"] = ok["actual_min"] > ok["promised_min"]
cols = ["distance_km", "items", "promised_min", "hour"]
train, test = train_test_split(ok, test_size=0.25, random_state=42)

## Урок 3. Решающее дерево изнутри

Дерево — модель, которую можно прочитать как инструкцию: «если обещали 30 минут и в заказе больше двух позиций — скорее опоздает». Разбираемся, как дерево выбирает вопросы, как его нарисовать, откуда берётся вероятность опоздания и какими настройками удержать дерево от переобучения.

Теория — на сайте курса: `ch06/03-tree.html`. Задачи «собери код из строк» (1) остались на сайте.

### Функция Джини · разминка

Напишите функцию `gini_p(p)`, которая по **доле** опозданий `p` (числу от 0 до 1) возвращает неоднородность Джини. Имя другое, чем у `gini(s)` из теории: та принимает столбец True/False и сама считает долю, а эта — сразу долю. Затем сохраните в `g_train` неоднородность всей обучающей выборки (3 знака).

*Подсказки (2): `dsc.hint('gini')`*

In [ ]:
%%task ch06/03-tree/gini
import pandas as pd
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
y = ok["actual_min"] > ok["promised_min"]
y_train, y_test = train_test_split(y, test_size=0.25, random_state=42)

### Качество одного вопроса · база

Посчитайте руками, насколько хорош вопрос `distance_km <= 2.0` на обучающей выборке. Сохраните `g_left` и `g_right` — неоднородность Джини левой (`<= 2.0`) и правой частей, и `g_split` — взвешенную неоднородность после вопроса (все — 3 знака). В `better` запишите `True`, если этот вопрос лучше вопроса про обещание (`promised_min <= 37.5`, взвешенная неоднородность 0.316), иначе `False`.

*Подсказки (2): `dsc.hint('split-gini')`*

In [ ]:
%%task ch06/03-tree/split-gini
import pandas as pd
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
train, test = train_test_split(ok, test_size=0.25, random_state=42)

def gini(s):
    p = s.mean()
    return 1 - p**2 - (1 - p)**2

### Вероятность опоздания · база

Обучите дерево глубины 3 и сохраните в `p_late` вероятности опоздания для всех тестовых доставок (массив из второго столбца `predict_proba`). Сохраните `n_risky` — сколько тестовых доставок имеют вероятность опоздания не меньше 0.6, и `n_values` — сколько **разных** значений встречается в `p_late`.

*Подсказки (2): `dsc.hint('ch06/03-tree/proba')`*

In [ ]:
%%task ch06/03-tree/proba
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
cols = ["distance_km", "items", "promised_min", "hour"]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(ok[cols], y, test_size=0.25, random_state=42)

### Нарисовать дерево · база

Обучите дерево с `max_leaf_nodes=4` на обучающей части и нарисуйте его через `plot_tree` с названиями признаков, подписями классов `"вовремя"` и `"опоздает"` и заливкой цветом. Сохраните число листьев в `n_leaves`.

*Подсказки (2): `dsc.hint('draw')`*

In [ ]:
%%task ch06/03-tree/draw
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
cols = ["distance_km", "items", "promised_min", "hour"]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(ok[cols], y, test_size=0.25, random_state=42)

### Минимум в листе · база

Обучите деревья **без** ограничения глубины, но с `min_samples_leaf` из списка `[1, 5, 20, 50, 100]`. Соберите словарь `test_by_leaf`: ключ — значение `min_samples_leaf`, значение — доля правильных ответов на тесте (3 знака). В `best_leaf` сохраните значение `min_samples_leaf` с лучшей долей.

*Подсказки (2): `dsc.hint('leaf-size')`*

In [ ]:
%%task ch06/03-tree/leaf-size
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
cols = ["distance_km", "items", "promised_min", "hour"]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(ok[cols], y, test_size=0.25, random_state=42)
test_by_leaf = {}

### Перебор порогов · посложнее

Повторите работу дерева для одного признака `distance_km`. Возьмите все уникальные значения расстояния в обучающей выборке по возрастанию, кандидаты в пороги — середины между соседними значениями. Для каждого порога посчитайте взвешенную неоднородность Джини. Сохраните `best_t` — порог с наименьшей неоднородностью и `best_g` — эту неоднородность (оба — 3 знака), а в `tree_t` — порог, который выбрало дерево `DecisionTreeClassifier(max_depth=1, random_state=0)`, обученное только на `distance_km` (3 знака).

*Подсказки (2): `dsc.hint('ch06/03-tree/best-threshold')`*

In [ ]:
%%task ch06/03-tree/best-threshold
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["late"] = ok["actual_min"] > ok["promised_min"]
train, test = train_test_split(ok, test_size=0.25, random_state=42)

def gini(s):
    p = s.mean()
    return 1 - p**2 - (1 - p)**2

### Флаг «пеший» · посложнее

Добавьте признак-флаг `walk` (1 — пеший курьер, 0 — нет) к признакам `distance_km`, `items`, `promised_min`, `hour` (флаг — последним столбцом). Обучите дерево глубины 3. Сохраните `top_feature` — признак с наибольшей важностью, `test_tree` — долю правильных ответов дерева на тесте и `test_rule` — долю правильных ответов правила «пеший — опоздает» на тех же тестовых доставках (обе — 3 знака).

*Подсказки (2): `dsc.hint('walk-flag')`*

In [ ]:
%%task ch06/03-tree/walk-flag
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]

### Вероятность своими руками · со звёздочкой

Докажите, что вероятность дерева — это доля опозданий в листе. Метод `model.apply(X)` возвращает для каждой строки номер листа, в который она попала. Обучите дерево глубины 3, посчитайте по **обучающей** выборке долю опозданий в каждом листе и с её помощью получите вероятности для тестовых доставок — массив или Series `p_manual` в порядке строк `X_test`. В `same` запишите `True`, если `p_manual` совпадает с `predict_proba(X_test)[:, 1]` с точностью до 1e-9.

*Подсказки (3): `dsc.hint('leaf-share')` · задача по желанию*

In [ ]:
%%task ch06/03-tree/leaf-share
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
cols = ["distance_km", "items", "promised_min", "hour"]
y = ok["actual_min"] > ok["promised_min"]
X_train, X_test, y_train, y_test = train_test_split(ok[cols], y, test_size=0.25, random_state=42)
model = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)

## Урок 4. Категориальные признаки и утечки

Главная причина опозданий — тип курьера, но `courier` — текст, а sklearn понимает только числа. В уроке «Решающее дерево изнутри» мы сделали вручную один флаг «пеший»; теперь учимся превращать в числа любые категории, готовить новые данные так же, как обучающие, и ловить утечки — признаки, которые подсказывают модели ответ из будущего.

Теория — на сайте курса: `ch06/04-categorical.html`. Задачи «собери код из строк» (1) остались на сайте.

### Флаги курьера · разминка

Закодируйте столбец `courier` флагами (0/1) и сохраните таблицу в `dm`. В `n_walk` сохраните число пеших доставок — сумму соответствующего флага.

*Подсказки (2): `dsc.hint('dummies')`*

In [ ]:
%%task ch06/04-categorical/dummies
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]

### Сколько столбцов · разминка

Закодируйте `courier`, `weather` и `shop_id` (только их) двумя способами: со всеми флагами и с `drop_first=True`. Сохраните число столбцов в `n_full` и `n_drop`.

*Подсказки (2): `dsc.hint('drop-first')`*

In [ ]:
%%task ch06/04-categorical/drop-first
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
cats = ok[["courier", "weather", "shop_id"]]

### Что дают погода и кофейня · база

В теории курьер поднял точность с 78 % до 90 %. А что добавляют другие категории? Признаки — `distance_km`, `items`, `promised_min`, `hour` и флаги `courier`. Обучите `DecisionTreeClassifier(max_depth=3, random_state=0)` трижды и сохраните долю правильных ответов на тесте (3 знака):

- `acc_courier` — числа и курьер;
- `acc_weather` — то же плюс флаги `weather`;
- `acc_shop` — числа, курьер и флаги `shop_id` (без погоды).

В `gain_weather` и `gain_shop` — прирост к `acc_courier` в процентных пунктах (1 знак). Разбиение — `train_test_split(..., test_size=0.25, random_state=42)` для всех вариантов. Прирост может оказаться нулевым — это тоже ответ.

*Подсказки (2): `dsc.hint('ch06/04-categorical/gain')`*

In [ ]:
%%task ch06/04-categorical/gain
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
num = ["distance_km", "items", "promised_min", "hour"]

### Коды против флагов · база

Сравните два способа подать модели курьера. Признаки — `distance_km`, `promised_min` и курьер:

- `acc_codes` — курьер одним столбцом `courier` с кодами `{"авто": 0, "пеший": 1, "вело": 2}` (порядок другой, чем в теории, но пеший снова в середине);
- `acc_flags` — курьер флагами `get_dummies`;
- `acc_lucky` — снова коды, но в другом порядке: `{"пеший": 0, "авто": 1, "вело": 2}`.

Модель — `DecisionTreeClassifier(max_depth=2, random_state=0)`, оценка на тесте (разбиение 0.25 / 42), доли — 3 знака.

*Подсказки (2): `dsc.hint('codes')`*

In [ ]:
%%task ch06/04-categorical/codes
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
y = ok["actual_min"] > ok["promised_min"]

### Заказы с новой погодой · база

Модель обучена на флагах признаков `distance_km`, `promised_min`, `courier`, `weather`. Подготовьте таблицу `X_new` для трёх новых заказов так, чтобы её столбцы совпадали со столбцами `X`, и сохраните предсказания списком `True/False` в `preds`.

*Подсказки (2): `dsc.hint('ch06/04-categorical/new-orders')`*

In [ ]:
%%task ch06/04-categorical/new-orders
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
X = pd.get_dummies(ok[["distance_km", "promised_min", "courier", "weather"]], dtype=int)
model = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X, ok["actual_min"] > ok["promised_min"])

new = pd.DataFrame({"distance_km": [1.5, 1.8, 7.5],
                    "promised_min": [30, 30, 60],
                    "courier": ["пеший", "вело", "авто"],
                    "weather": ["гроза", "снег", "ясно"]})

### Оценка гостя · база

Проверьте, можно ли брать `rating` в признаки.

- `late_by_rating` — доля опозданий для каждой оценки (Series: индекс — оценка, значения округлите до 2 знаков);
- `leaks` — список столбцов из `["distance_km", "rating", "weather", "actual_min", "items", "courier"]`, которые **нельзя** брать в признаки для прогноза опоздания в момент отправки курьера.

*Подсказки (2): `dsc.hint('rating-leak')`*

In [ ]:
%%task ch06/04-categorical/rating-leak
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
late = ok["actual_min"] > ok["promised_min"]

### Подозрительно хорошо · посложнее

Коллега добавил к признакам столбец `delay` = `actual_min − promised_min` («отклонение от обещания») и радуется результату. Повторите его модель: флаги признаков `distance_km`, `items`, `promised_min`, `courier`, `weather` плюс `delay`, разбиение 0.25 / 42, `DecisionTreeClassifier(max_depth=3, random_state=0)`.

Сохраните `acc_test` — долю правильных ответов на тесте (3 знака) и `top` — название признака с наибольшей важностью.

*Подсказки (2): `dsc.hint('suspicious')`*

In [ ]:
%%task ch06/04-categorical/suspicious
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)].copy()
ok["delay"] = ok["actual_min"] - ok["promised_min"]
y = ok["actual_min"] > ok["promised_min"]

### Доля опозданий кофейни без подглядывания · посложнее

Сделайте признак «доля опозданий в кофейне» честно. Разбейте `ok[["shop_id"]]` и `y` (0.25 / 42). Посчитайте долю опозданий по кофейням **только по обучающей части** — `shop_rate` (Series: индекс — `shop_id`). Добавьте в `X_train` и `X_test` столбец `shop_rate` с этой долей для кофейни строки.

Сохраните также `diff` — наибольшее по модулю расхождение (по всем кофейням) между долей по обучающей части и долей по всем данным, в п.п., 2 знака.

*Подсказки (2): `dsc.hint('shop-rate')`*

In [ ]:
%%task ch06/04-categorical/shop-rate
import pandas as pd
from sklearn.model_selection import train_test_split
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]
y = ok["actual_min"] > ok["promised_min"]

### Одна подготовка для всех данных · со звёздочкой

Напишите функцию `prepare(df, columns=None)`, которая готовит признаки из таблицы доставок в формате `deliveries.csv`:

- добавляет `hour` — час из `created` (в `df` он строкой или датой — функция должна работать с обоими вариантами);
- берёт столбцы `distance_km`, `items`, `promised_min`, `hour`, `courier`, `weather` и кодирует их флагами (`dtype=int`);
- если передан `columns` — выравнивает результат по этим столбцам (недостающие — нулями, лишние — убрать).

Проверка вызовет `prepare` на обучающих данных, а затем на новых заказах с `columns=` столбцами обучения — и сравнит результат с эталоном. Исходную таблицу функция менять не должна.

*Подсказки (3): `dsc.hint('ch06/04-categorical/prepare')` · задача по желанию*

In [ ]:
%%task ch06/04-categorical/prepare
import pandas as pd
d = pd.read_csv("deliveries.csv")

def prepare(df, columns=None):
    ...

X = prepare(d)
X.head()

## Урок 5. Случайный лес

Одно глубокое дерево запоминает обучающие данные вместе со случайным шумом. Но если вырастить сотню разных деревьев и дать им проголосовать, их ошибки во многом взаимно гасятся. Так устроен случайный лес — одна из самых надёжных моделей для табличных данных. Строим лес сначала руками, потом в sklearn, и честно сравниваем его с правилом аналитика.

Теория — на сайте курса: `ch06/05-forest.html`. Задачи «собери код из строк» (1) остались на сайте.

### Бутстрэп-выборка · разминка

Сделайте из `ok` бутстрэп-выборку `boot` (`random_state=0`) и сохраните в `share_unique` долю разных доставок (`delivery_id`) в ней от числа строк `ok` (3 знака).

*Подсказки (2): `dsc.hint('ch06/05-forest/boot')`*

In [ ]:
%%task ch06/05-forest/boot
import pandas as pd
d = pd.read_csv("deliveries.csv")
ok = d[d["actual_min"].between(1, 299)]

### Лес на всех признаках · разминка

Обучите `RandomForestClassifier(n_estimators=50, random_state=0)` на `X_train` и сохраните модель в `forest`, а долю правильных ответов на тесте — в `acc_test` (3 знака).

*Подсказки (2): `dsc.hint('first-forest')`*

In [ ]:
%%task ch06/05-forest/first-forest
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test

### Кто переобучился сильнее · база

Обучите на одних и тех же данных дерево без ограничения глубины (`DecisionTreeClassifier(random_state=0)`) и лес `RandomForestClassifier(n_estimators=50, random_state=0)`. Для каждой модели посчитайте разрыв «обучение − тест» в процентных пунктах (1 знак): `gap_tree` и `gap_forest`. В `winner` сохраните `"лес"` или `"дерево"` — у кого выше доля правильных ответов на тесте.

*Подсказки (2): `dsc.hint('gap')`*

In [ ]:
%%task ch06/05-forest/gap
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test

### Кривая числа деревьев · база

Для числа деревьев из списка `sizes` обучите `RandomForestClassifier(n_estimators=n, random_state=0)` и сохраните доли правильных ответов на тесте в список `accs` (без округления). Нарисуйте линию: по оси X — `sizes`, по Y — `accs`, подпишите ось X `"число деревьев"`.

*Подсказки (2): `dsc.hint('n-trees')`*

In [ ]:
%%task ch06/05-forest/n-trees
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test
# sizes = [1, 3, 10, 30, 60] — список чисел деревьев

### Главные признаки леса · база

Обучите `RandomForestClassifier(n_estimators=50, min_samples_leaf=5, random_state=0)` на `X_train`. Сохраните `imp` — Series важностей признаков (индекс — названия столбцов), отсортированную по убыванию, и `top3` — список названий трёх самых важных признаков. В `courier_share` — суммарную важность трёх флагов курьера (3 знака).

*Подсказки (2): `dsc.hint('ch06/05-forest/importance')`*

In [ ]:
%%task ch06/05-forest/importance
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test

### Ансамбль своими руками · база

Повторите ручной ансамбль из теории, но из 7 деревьев: дерево `i` (от 0 до 6) — `DecisionTreeClassifier(random_state=0)` на бутстрэп-выборке `X_train.sample(frac=1, replace=True, random_state=i)`. Сохраните:

- `tree_accs` — список долей правильных ответов каждого дерева на тесте;
- `acc_vote` — долю правильных ответов голосования большинством (3 знака);
- `acc_forest7` — долю правильных ответов настоящего леса из семи деревьев: `RandomForestClassifier(n_estimators=7, random_state=0)` (3 знака). Деревья у него другие, не ваши: лес сам делает свои бутстрэп-выборки и на каждом вопросе выбирает случайные признаки, поэтому результат не обязан совпасть с `acc_vote`.

*Подсказки (2): `dsc.hint('ch06/05-forest/vote')`*

In [ ]:
%%task ch06/05-forest/vote
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test

### Кто выигрывает стабильно · посложнее

В теории лес обыгрывал правило «пеший — опоздает» на каждом из пяти разбиений. А простое дерево? Сравните три модели на разбиениях с `random_state` от 0 до 4 (`test_size=0.25`): лес `RandomForestClassifier(n_estimators=50, random_state=0)`, дерево `DecisionTreeClassifier(max_depth=3, random_state=0)` и правило.

Сохраните таблицу `cmp` (индекс — номер разбиения 0–4) со столбцами `forest`, `tree` и `rule` — доли правильных ответов на тесте, а также `forest_always` и `tree_always` — `True`, если модель обыграла правило на **всех** пяти разбиениях.

*Подсказки (2): `dsc.hint('vs-rule')`*

In [ ]:
%%task ch06/05-forest/vs-rule
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X

### Где лес спорит с правилом · посложнее

Лес (`n_estimators=50, random_state=0`, разбиение 0.25 / 42) и правило «пеший — опоздает» отвечают одинаково на большинстве тестовых доставок. Найдите доставки, где ответы **различаются**, и сохраните:

- `n_disagree` — сколько таких доставок в тесте;
- `forest_right` — на скольких из них прав лес;
- `walk_share` — доля пеших среди них (3 знака).

*Подсказки (2): `dsc.hint('disagree')`*

In [ ]:
%%task ch06/05-forest/disagree
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test, forest

### Лес — это среднее деревьев · со звёздочкой

Убедитесь, что вероятности леса — среднее вероятностей его деревьев. Обученные деревья леса лежат в списке `forest.estimators_`. Сохраните:

- `manual` — массив вероятности опоздания для `X_test`, усреднённый по всем деревьям леса (столбец 1 из `predict_proba` каждого дерева);
- `max_diff` — наибольшее абсолютное расхождение `manual` с `forest.predict_proba(X_test)[:, 1]`.

*Подсказки (2): `dsc.hint('avg-proba')` · задача по желанию*

In [ ]:
%%task ch06/05-forest/avg-proba
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# данные подготовлены, как в теории: d, ok, y, X, X_train, X_test, y_train, y_test
# forest — обученный лес из 30 деревьев (n_estimators=30)

## Урок 6. Логистическая регрессия и вероятности

Диспетчеру мало ответа «опоздает / не опоздает»: ему нужно знать, насколько рискован заказ, чтобы позвонить гостю заранее или отдать заказ вело-курьеру. Логистическая регрессия выдаёт вероятность опоздания. Учимся получать вероятности, выбирать порог тревоги, читать коэффициенты. Ранжирование по риску и проверка, можно ли вероятностям верить, — в следующем уроке.

Теория — на сайте курса: `ch06/06-logistic.html`. Задачи «собери код из строк» (1) остались на сайте.

### Сигмоида · разминка

Напишите функцию `sigmoid(z)`, которая возвращает 1 / (1 + e⁻ᶻ) (для e используйте `math.exp`). Сохраните в `probs` список значений сигмоиды для z = −2, 0, 3, округлённых до 3 знаков.

*Подсказки (2): `dsc.hint('sigmoid')`*

In [ ]:
%%task ch06/06-logistic/sigmoid
import math

### Вероятность опоздания · разминка

Обучите `LogisticRegression(max_iter=2000)` на подготовленных признаках. Сохраните в `p` вероятности опоздания для тестовых доставок (одномерный массив — столбец `[:, 1]` из `predict_proba`), в `mean_p` — среднюю вероятность (3 знака), в `n_alarm` — сколько тестовых доставок получили вероятность 0.5 и выше (целое число).

*Подсказки (2): `dsc.hint('ch06/06-logistic/proba')`*

In [ ]:
%%task ch06/06-logistic/proba
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Порог под требование диспетчера · база

Диспетчер решил поднимать тревогу, если вероятность опоздания 0.3 и выше. Для тестовых доставок посчитайте (все — 3 знака):

- `alarms` — долю доставок с тревогой;
- `caught` — долю настоящих опозданий, на которые была тревога;
- `hit` — долю тревог, которые оправдались (доставка опоздала).

Затем диспетчер требует, чтобы оправдывались не меньше 80 % тревог. Переберите пороги `np.arange(0.10, 0.90, 0.05).round(2)` и сохраните в `t80` первый (самый низкий) порог, при котором `hit` ≥ 0.8, а в `caught80` — долю пойманных опозданий при нём (3 знака).

*Подсказки (3): `dsc.hint('threshold')`*

In [ ]:
%%task ch06/06-logistic/threshold
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
p = LogisticRegression(max_iter=2000).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Знаки коэффициентов · база

Сохраните в `coefs` веса модели — Series с названиями признаков в индексе, отсортированную по возрастанию. В `top_up` — название признака с самым большим положительным весом, в `top_down` — с самым отрицательным. В `snow_sign` — строку `"+"` или `"-"`: повышает или понижает снег риск опоздания по мнению модели.

*Подсказки (2): `dsc.hint('ch06/06-logistic/coefs')`*

In [ ]:
%%task ch06/06-logistic/coefs
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LogisticRegression(max_iter=2000).fit(X_train, y_train)

### Риск для новых заказов · посложнее

Оцените вероятность опоздания для трёх новых заказов. Признаки новых заказов нужно привести к столбцам обучающей таблицы, как в уроке «Категориальные признаки и утечки». Сохраните в `risk` **список** из трёх вероятностей опоздания (2 знака) в порядке заказов.

| distance_km | items | promised_min | hour | courier | weather | shop_id |
|---|---|---|---|---|---|---|
| 1.5 | 3 | 30 | 18 | пеший | снег | KZN-2 |
| 1.5 | 3 | 30 | 18 | вело | снег | KZN-2 |
| 6.0 | 2 | 45 | 12 | авто | ясно | PRM-1 |

*Подсказки (2): `dsc.hint('new-risk')`*

In [ ]:
%%task ch06/06-logistic/new-risk
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LogisticRegression(max_iter=2000).fit(X_train, y_train)

new = pd.DataFrame({
    "distance_km": [1.5, 1.5, 6.0], "items": [3, 3, 2], "promised_min": [30, 30, 45], "hour": [18, 18, 12],
    "courier": ["пеший", "вело", "авто"], "weather": ["снег", "снег", "ясно"], "shop_id": ["KZN-2", "KZN-2", "PRM-1"],
})

## Урок 7. Что делать с вероятностями: ранжирование и калибровка

Логистическая регрессия выдаёт вероятность. Ею можно не только ставить порог: можно отсортировать заказы по риску и работать с верхом списка — и проверить, можно ли самой вероятности верить как числу. Ранжирование, лифт и калибровка — то, чем аналитик пользуется чаще, чем accuracy.

Теория — на сайте курса: `ch06/06b-probabilities.html`.

### Лифт и охват на пальцах · разминка

Отдел лояльности разослал предложение 2 000 гостям, купили 100. Модель заранее отобрала 200 гостей с наибольшей вероятностью покупки, и среди них купили 40. Посчитайте обычной арифметикой: `top_rate` — долю купивших в отобранных 200, `all_rate` — долю купивших среди всех 2 000, `lift` — во сколько раз первая больше второй, `captured` — какую долю всех покупателей накрыл список из 200 (охват). Округлять не нужно.

*Подсказки (2): `dsc.hint('lift-hand')`*

In [ ]:
%%task ch06/06b-probabilities/lift-hand
total, buyers = 2000, 100       # все гости и все купившие
top, top_buyers = 200, 40       # отобранные моделью и купившие среди них

### Топ-10 % по риску · база

Возьмите 10 % тестовых доставок с наибольшей вероятностью опоздания (`k = int(len(...) * 0.1)` штук). Сохраните `top_rate` — долю опозданий среди них (3 знака), `all_rate` — долю опозданий во всём тесте (3 знака) и `lift` — во сколько раз первая больше второй (2 знака, считайте по неокруглённым долям).

*Подсказки (2): `dsc.hint('top10')`*

In [ ]:
%%task ch06/06b-probabilities/top10
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
p = LogisticRegression(max_iter=2000).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Список на 20 % заказов · база

У диспетчера стало вдвое больше времени: теперь он может проконтролировать 20 % тестовых доставок (`k = int(len(...) * 0.2)` штук с наибольшей вероятностью опоздания). Сохраните `top_rate` — долю опозданий в этом списке, `lift` — во сколько раз она больше средней доли опозданий в тесте и `captured` — охват: какую долю всех опозданий теста накрывает список (все — 3 знака, считайте по неокруглённым долям).

*Подсказки (2): `dsc.hint('capture')`*

In [ ]:
%%task ch06/06b-probabilities/capture
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
p = LogisticRegression(max_iter=2000).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Риск внутри вело-доставок · со звёздочкой

Правило «пеший — опоздает» всем вело-доставкам отвечает одинаково: «вовремя». Проверьте, умеет ли модель различать риск **внутри** вело-доставок. Отберите тестовые доставки вело-курьеров, отсортируйте их по вероятности опоздания и разделите пополам: в `high_rate` — доля опозданий в более рискованной половине, в `low_rate` — в менее рискованной (обе — 3 знака). Если таких доставок нечётное число, лишняя доставка идёт в менее рискованную половину: `half = n // 2` верхних — это рискованная половина.

*Подсказки (2): `dsc.hint('bikes')` · задача по желанию*

In [ ]:
%%task ch06/06b-probabilities/bikes
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
p = LogisticRegression(max_iter=2000).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Калибровка · посложнее

Проверьте калибровку модели на тесте. Разбейте вероятности на группы `pd.cut(..., [0, 0.25, 0.5, 0.75, 1])` и сохраните в `cal` таблицу с индексом-группой и тремя столбцами: `pred` — средняя вероятность в группе, `fact` — фактическая доля опозданий, `n` — число доставок. Пустые группы не нужны. В `worst` сохраните наибольший модуль расхождения `|pred − fact|` среди групп — число ≥ 0, 3 знака.

*Подсказки (2): `dsc.hint('ch06/06b-probabilities/calib')`*

In [ ]:
%%task ch06/06b-probabilities/calib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
p = LogisticRegression(max_iter=2000).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Самоуверенное дерево · посложнее

Проверьте предупреждение из теории: обучите на тех же данных дерево без ограничений `DecisionTreeClassifier(random_state=0)` и получите его вероятности опоздания для теста `tp = ....predict_proba(X_test)[:, 1]`. Сохраните (все — 3 знака):

- `extreme` — долю тестовых доставок, у которых вероятность дерева ровно 0 или ровно 1;
- `fact_one` — фактическую долю опозданий среди доставок, которым дерево дало вероятность 1;
- `fact_zero` — фактическую долю опозданий среди доставок с вероятностью 0.

*Подсказки (2): `dsc.hint('deep-calib')`*

In [ ]:
%%task ch06/06b-probabilities/deep-calib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"] > ok["promised_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## Урок 8. Регрессия: сколько минут

«Опоздает или нет» — грубый ответ. Гостю полезнее услышать «привезём примерно через 42 минуты». Предсказываем число — фактическое время доставки: линейная регрессия, дерево и лес для чисел, метрики MAE, RMSE и R², разбор ошибок. И увидим, где машинное обучение наконец заметно обгоняет правило аналитика.

Теория — на сайте курса: `ch06/07-regression.html`. Задачи «собери код из строк» (1) остались на сайте.

### MAE и RMSE для пяти доставок · разминка

Модель предсказала время пяти доставок. Посчитайте `mae` и `rmse` (обе — 2 знака) функциями sklearn `mean_absolute_error` и `mean_squared_error`.

*Подсказки (2): `dsc.hint('mae-hand')`*

In [ ]:
%%task ch06/07-regression/mae-hand
from sklearn.metrics import mean_absolute_error, mean_squared_error
fact = [30, 42, 25, 55, 38]
pred = [33, 40, 25, 45, 36]

### Базовая модель на тесте · разминка

Посчитайте базовую модель для регрессии честно: медиану возьмите **по обучающей части**, а ошибку — на тестовой. Сохраните `base` — медиану `y_train` и `mae_base` — MAE постоянного прогноза `base` на тесте (2 знака).

*Подсказки (2): `dsc.hint('base-median')`*

In [ ]:
%%task ch06/07-regression/base-median
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Линейная регрессия на всех признаках · база

Обучите `LinearRegression` на `X_train`. Сохраните модель в `lin`, `mae_lin` — MAE на тесте (2 знака), `r2_lin` — R² на тесте (3 знака), `per_km` — вес признака `distance_km` (2 знака) и `walk_vs_auto` — на сколько минут, по мнению модели, пеший курьер медленнее авто при прочих равных (разница весов, 1 знак).

*Подсказки (2): `dsc.hint('linear')`*

In [ ]:
%%task ch06/07-regression/linear
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Три метрики для леса · база

Обучите `RandomForestRegressor(n_estimators=50, random_state=0)` и посчитайте на тесте `mae_rf`, `rmse_rf` (2 знака) и `r2_rf` (3 знака). В `gain` сохраните, на сколько минут MAE леса меньше MAE медианы обучающей части (2 знака, считайте по неокруглённым значениям).

*Подсказки (2): `dsc.hint('forest-metrics')`*

In [ ]:
%%task ch06/07-regression/forest-metrics
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Глубина дерева для минут · база

Обучите `DecisionTreeRegressor(random_state=0)` с глубиной 2, 5 и без ограничения (`max_depth=None`). Сохраните в `mae_by_depth` словарь `{2: ..., 5: ..., None: ...}` с MAE на тесте (2 знака), а в `best_depth` — глубину с наименьшей MAE.

*Подсказки (2): `dsc.hint('ch06/07-regression/depth')`*

In [ ]:
%%task ch06/07-regression/depth
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### График «факт против прогноза» · база

Нарисуйте для линейной регрессии точечный график: по оси X — прогноз на тесте, по оси Y — факт. Добавьте серую пунктирную диагональ `ax.plot([10, 100], [10, 100], ...)`, подписи осей `прогноз, мин` и `факт, мин` и заголовок, в котором есть слово `линейная`.

*Подсказки (2): `dsc.hint('fact-pred')`*

In [ ]:
%%task ch06/07-regression/fact-pred
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
pred = LinearRegression().fit(X_train, y_train).predict(X_test)

### Где ошибается лес · посложнее

Для леса `RandomForestRegressor(n_estimators=50, random_state=0)` соберите таблицу `res` со столбцами `fact`, `pred` и `err` (факт − прогноз) по тестовым доставкам и добавьте к ней столбец `courier` из `ok` (через `join`). Сохраните:

- `big_share` — долю доставок, где модель ошиблась больше чем на 10 минут в любую сторону (3 знака);
- `err_by_courier` — Series: средний остаток по типу курьера (индекс — тип курьера, значения — 2 знака);
- `worst_courier` — тип курьера в строке с самой большой по модулю ошибкой.

*Подсказки (2): `dsc.hint('ch06/07-regression/residuals')`*

In [ ]:
%%task ch06/07-regression/residuals
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
forest = RandomForestRegressor(n_estimators=50, random_state=0).fit(X_train, y_train)

### Опоздание из линейной модели · посложнее

Превратите прогноз минут **линейной регрессии** в прогноз опоздания: «прогноз > обещания». Сохраните `acc_lin` — долю правильных ответов на тесте (3 знака), `late_minutes` — на сколько минут в среднем опаздывают те тестовые доставки, которые модель пометила как опаздывающие (среднее `факт − обещание` по ним, 1 знак; учитываются все такие доставки, даже приехавшие вовремя).

*Подсказки (2): `dsc.hint('late-minutes')`*

In [ ]:
%%task ch06/07-regression/late-minutes
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
pred = LinearRegression().fit(X_train, y_train).predict(X_test)

### Помочь линейной модели · со звёздочкой

Линейная модель не видит, что пешему курьеру километр стоит дороже. Помогите ей: добавьте в `X` признак-взаимодействие `walk_km` — расстояние, умноженное на флаг пешего курьера (у остальных курьеров 0). Обучите `LinearRegression` заново на том же разбиении и сохраните `mae_before` и `mae_after` — MAE на тесте без нового признака и с ним (2 знака), а также `walk_km_coef` — вес нового признака (2 знака).

*Подсказки (3): `dsc.hint('interaction')` · задача по желанию*

In [ ]:
%%task ch06/07-regression/interaction
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("deliveries.csv", parse_dates=["created"])
ok = d[d["actual_min"].between(1, 299)].copy()
ok["hour"] = ok["created"].dt.hour
y = ok["actual_min"]
X = pd.get_dummies(ok[["distance_km", "items", "promised_min", "hour", "courier", "weather", "shop_id"]], dtype=int)

## Урок 9. Честное сравнение моделей

У нас уже есть правило аналитика, дерево, лес и логистическая регрессия. Какая модель лучше — и насколько можно верить цифре? Учимся подбирать настройки так, чтобы не обманывать себя: делим данные на три части, не подглядываем в тест и сравниваем модели кросс-валидацией. Групповое разбиение и время обучения — в следующем уроке.

Теория — на сайте курса: `ch06/08-validation.html`. Задачи «собери код из строк» (1) остались на сайте.

### Три части · разминка

`X` (признаки с `get_dummies`) и `y` (опоздание) уже готовы — как в теории. Разделите данные на три части 60/20/20:

1. `train_test_split(X, y, test_size=0.2, random_state=42)` → `X_rest`, `X_test`, `y_rest`, `y_test`;
2. `train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)` → `X_train`, `X_valid`, `y_train`, `y_valid`.

*Подсказки (2): `dsc.hint('three-parts')`*

In [ ]:
%%task ch06/08-validation/three-parts
from sklearn.model_selection import train_test_split
# X и y уже готовы

### Размеры фолдов · разминка

`X` уже готов. Нарежьте его `KFold(n_splits=5, shuffle=True, random_state=42)`. Сохраните в `sizes` список размеров проверочных частей (по одному числу на фолд) и в `total` — их сумму. Сравните `total` с числом строк `X`.

*Подсказки (2): `dsc.hint('ch06/08-validation/folds')`*

In [ ]:
%%task ch06/08-validation/folds
from sklearn.model_selection import KFold

### Настройка леса по валидации · база

Данные уже разделены на `X_train`, `X_valid`, `X_test` (и `y_…`) — как в первой задаче. Повторите приём из теории для другой модели и другой настройки: лес `RandomForestClassifier(n_estimators=50, min_samples_leaf=leaf, random_state=0)` для `leaf` из `[1, 5, 20, 50]`. Обучайте на обучающей части и сохраните словарь `valid_acc`: `leaf` → доля правильных на валидации. В `best_leaf` — значение с лучшей валидацией (при равенстве — меньшее). Обучите лес с этим `min_samples_leaf` на обучающей части и **один раз** посчитайте `test_acc` — долю правильных на тесте (3 знака).

*Подсказки (2): `dsc.hint('pick-depth')`*

In [ ]:
%%task ch06/08-validation/pick-depth
from sklearn.ensemble import RandomForestClassifier
# X_train, X_valid, X_test, y_train, y_valid, y_test уже готовы
valid_acc = {}

### Короткое и глубокое дерево · база

`X`, `y` и `kf = KFold(n_splits=5, shuffle=True, random_state=42)` уже готовы. Оцените кросс-валидацией два дерева (`random_state=0`): глубины 3 и без ограничения глубины. Сохраните средние `mean3`, `mean_full` и разбросы (std) `std3`, `std_full` — всё с 3 знаками.

*Подсказки (2): `dsc.hint('cv-depth')`*

In [ ]:
%%task ch06/08-validation/cv-depth
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

### Правило на тех же фолдах · база

`ok`, `y` и `kf` уже готовы. Посчитайте долю правильных ответов правила «пеший — опоздает» на каждом проверочном фолде `kf.split(X)`. Сохраните список `rule_scores` (5 чисел, 3 знака) и среднее `rule_mean` (3 знака). Так же для «всегда вовремя» — только среднее `base_mean` (3 знака).

*Подсказки (2): `dsc.hint('rule-cv')`*

In [ ]:
%%task ch06/08-validation/rule-cv
import numpy as np
rule = ok["courier"] == "пеший"

### Итоговая таблица с отложенным тестом · база

`X`, `y`, `ok` и `kf` уже готовы. Сделайте сравнение по рабочему протоколу:

1. Отложите тест: `train_test_split(X, y, test_size=0.2, random_state=42)` → `X_rest`, `X_test`, `y_rest`, `y_test`.
2. На фолдах `kf` по `X_rest` сравните правило и три модели:
   - `"правило"` — «пеший — опоздает»: флаг `courier_пеший` в `X_rest`;
   - `"дерево"` — `DecisionTreeClassifier(max_depth=3, random_state=0)`;
   - `"лес"` — `RandomForestClassifier(n_estimators=50, random_state=0)`;
   - `"логрегрессия"` — `LogisticRegression(max_iter=2000)`.
3. Сохраните таблицу `table`: индекс — эти названия, столбцы `mean` и `std` (3 знака), строки по убыванию `mean`. В `winner` — название лучшей строки.
4. Один раз проверьте победителя на тесте: обучите его на всём `X_rest` и сохраните долю правильных на `X_test` в `winner_test` (3 знака), а долю правила на тесте — в `rule_test` (3 знака).

*Подсказки (2): `dsc.hint('table')`*

In [ ]:
%%task ch06/08-validation/table
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

### Цена подглядывания · посложнее

`y_valid` и `y_test` уже готовы (разбиение 60/20/20 из теории). Повторите опыт с монетками, но с другим зерном: `rng = np.random.default_rng(7)`, 300 монеток. Для каждой монетки **по порядку**: сначала случайные ответы для валидации `rng.random(len(y_valid)) < 0.5`, потом для теста `rng.random(len(y_test)) < 0.5`. Сохраните:

- `best_valid` — лучшую долю правильных на валидации (3 знака);
- `its_test` — долю правильных **этой же** монетки на тесте (3 знака);
- `gap` — `best_valid − its_test` в п.п. (1 знак).

*Подсказки (2): `dsc.hint('peek')`*

In [ ]:
%%task ch06/08-validation/peek
import numpy as np
rng = np.random.default_rng(7)

### Две настройки и отложенный тест · посложнее

`X` и `y` уже готовы. Отложите тест: `train_test_split(X, y, test_size=0.2, random_state=42)` → `X_rest`, `X_test`, `y_rest`, `y_test`. Валидационную часть отдельно не выделяйте — вместо неё кросс-валидация на `X_rest` с `kf = KFold(n_splits=5, shuffle=True, random_state=42)`.

Переберите деревья `DecisionTreeClassifier(max_depth=md, min_samples_leaf=ml, random_state=0)` для `md` из `[2, 3, 4, 5, 6]` и `ml` из `[1, 20, 50]`. Сохраните:

- `cv_scores` — словарь `(md, ml)` → средняя доля правильных по фолдам;
- `best_params` — кортеж `(md, ml)` с наибольшим средним;
- `test_acc` — доля правильных на `X_test` дерева с лучшими настройками, обученного на всём `X_rest` (3 знака).

*Подсказки (2): `dsc.hint('grid-cv')`*

In [ ]:
%%task ch06/08-validation/grid-cv
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
cv_scores = {}

### Лес против правила: устойчивость · со звёздочкой

`X`, `y` и `ok` уже готовы. Отрыв леса от правила «пеший» — 1–2 п.п. Не случайность ли это нарезки? Повторите сравнение 10 раз, меняя только нарезку: `KFold(n_splits=5, shuffle=True, random_state=rs)` для `rs` от 0 до 9. В каждом повторе посчитайте среднюю по фолдам долю правильных у `RandomForestClassifier(n_estimators=30, random_state=0)` и у правила — на **одних и тех же** фолдах.

Сохраните `gaps` — список из 10 разниц «лес − правило» в п.п. (1 знак), `wins` — в скольких повторах лес лучше правила, и `mean_gap` — среднюю разницу (п.п., 1 знак).

*Подсказки (2): `dsc.hint('repeat-cv')` · задача по желанию*

In [ ]:
%%task ch06/08-validation/repeat-cv
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
rule = ok["courier"] == "пеший"

## Урок 10. Групповое разбиение и цена модели

Две проверки, без которых сравнение моделей нечестно. Если один гость встречается во многих строках, модель может запомнить его, а не выучить закономерность, — и обычная кросс-валидация этого не заметит. А если лучшая модель учится в десятки раз дольше, это тоже часть её цены.

Теория — на сайте курса: `ch06/08b-groups-speed.html`. Задачи «собери код из строк» (1) остались на сайте.

### Знакомые гости в проверке · разминка

Повторите игрушечный пример из теории и посчитайте, сколько раз гость из проверочной части уже встречался модели в обучении. Для каждого фолда возьмите множество гостей проверки и множество гостей обучения и сложите размеры их пересечений по всем фолдам: `shared_plain` — для `KFold(3, shuffle=True, random_state=0)`, `shared_group` — для `GroupKFold(3)` с `groups=toy["guest"]`.

*Подсказки (2): `dsc.hint('toy-folds')`*

In [ ]:
%%task ch06/08b-groups-speed/toy-folds
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold

toy = pd.DataFrame({"guest": list("AAAABBBCCC")})

### Новые гости в тесте · база

Отложите тест по гостям: `GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=4)` по `guest_orders.csv` с группами `guest_id`. Сохраните `n_test_rows` — число строк в тесте, `n_test_guests` — число гостей в тесте и `common` — сколько гостей есть и в обучении, и в тесте. Для сравнения сделайте обычный `train_test_split(..., test_size=0.25, random_state=4)` и сохраните `common_plain` — сколько гостей оказалось в обеих частях там.

*Подсказки (3): `dsc.hint('group-split')`*

In [ ]:
%%task ch06/08b-groups-speed/group-split
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
o = pd.read_csv("guest_orders.csv")
X = o[["weekday", "hour", "items", "amount", "distance_km"]]
y, groups = o["dessert"], o["guest_id"]

### Победитель зависит от разбиения · посложнее

Сравните на `guest_orders.csv` лес `RandomForestClassifier(n_estimators=50, random_state=0)` и `LogisticRegression(max_iter=2000)` двумя кросс-валидациями: `KFold(5, shuffle=True, random_state=42)` и `GroupKFold(5)` с группами `guest_id`. Сохраните `gap_forest` и `gap_lr` — насколько среднее accuracy при обычном разбиении выше, чем при групповом (3 знака), и `winner_plain`, `winner_group` — строку `"лес"` или `"логрегрессия"`: у кого выше среднее при каждом способе.

*Подсказки (3): `dsc.hint('group-cv')`*

In [ ]:
%%task ch06/08b-groups-speed/group-cv
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
o = pd.read_csv("guest_orders.csv")
X = o[["weekday", "hour", "items", "amount", "distance_km"]]
y, groups = o["dessert"], o["guest_id"]

### Детектор памяти · посложнее

В типичных ошибках сказано: разница обычной и групповой оценки — детектор признака, по которому модель «узнаёт» гостя. Проверьте это на `guest_orders.csv`: уберите из признаков `distance_km` (останутся `weekday`, `hour`, `items`, `amount`) и для леса `RandomForestClassifier(n_estimators=50, random_state=0)` посчитайте средние accuracy при `KFold(5, shuffle=True, random_state=42)` и при `GroupKFold(5)` с группами `guest_id`. Сохраните `plain_nod` и `group_nod` — эти средние (3 знака) и `gap_nod` — их разницу «обычное минус групповое» (3 знака, по неокруглённым средним).

*Подсказки (2): `dsc.hint('no-distance')`*

In [ ]:
%%task ch06/08b-groups-speed/no-distance
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
o = pd.read_csv("guest_orders.csv")
y, groups = o["dessert"], o["guest_id"]

### Качество и скорость · база

`X_train`, `X_test`, `y_train`, `y_test` уже готовы. Сравните дерево `DecisionTreeClassifier(max_depth=3, random_state=0)`, лес `RandomForestClassifier(n_estimators=100, random_state=0)` и логистическую регрессию `LogisticRegression(max_iter=2000)`. Соберите таблицу `speed`: индекс — `"дерево"`, `"лес"` и `"логрегрессия"`, столбцы `accuracy` (доля правильных на тесте, 3 знака), `fit_s` и `predict_s` (секунды, замер `time.perf_counter()`). В `fit_ratio` сохраните, во сколько раз лес учится дольше дерева.

*Подсказки (3): `dsc.hint('ch06/08b-groups-speed/speed-table')`*

In [ ]:
%%task ch06/08b-groups-speed/speed-table
import time
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
# готово, как в теории: X_train, X_test, y_train, y_test

## Урок 11. Проект: кто из гостей уйдёт

Финал главы. Отдел лояльности хочет вернуть гостей, которые вот-вот перестанут ходить в «Зерно», — но бонусы стоят денег, и раздавать их всем подряд нельзя. Строим модель оттока от осмотра данных до списка гостей для акции: ищем утечку, подбираем модель честной валидацией и считаем, насколько модель лучше правила.

Теория — на сайте курса: `ch06/09-project.html`.

### Шаг 1. Осмотр · база

Загрузите `guests_churn.csv` в `g`. Сохраните:

- `churn_rate` — долю ушедших гостей (3 знака);
- `na_check` — сколько пропусков в `avg_check`;
- `na_no_visits` — `True`, если **у всех** гостей с пропущенным `avg_check` `visits_90d == 0`;
- `churn_no_visits` — долю ушедших среди гостей без визитов за 90 дней (3 знака).

*Подсказки (2): `dsc.hint('ch06/09-project/look')`*

In [ ]:
%%task ch06/09-project/look
import pandas as pd

### Шаг 2. Пропуски среднего чека · база

`g` уже загружена. Средний чек пуст у гостей без визитов: делить сумму покупок на ноль визитов нельзя. Заполнять его средним или медианой — значит сказать модели, что эти гости тратят как обычные, хотя они не потратили ничего. Честное значение — 0.

Заполните пропуски `avg_check` нулём **в `g`** и сохраните `filled` — сколько значений заполнили. Затем сравните средний чек ушедших и оставшихся среди гостей **с визитами**: `check_churned` и `check_stayed` (0 знаков). Разница может оказаться маленькой — это тоже ответ.

*Подсказки (2): `dsc.hint('ch06/09-project/missing')`*

In [ ]:
%%task ch06/09-project/missing
# g — исходная таблица

### Шаг 3. Подозрительно хорошо · посложнее

`g` с заполненным `avg_check` уже готова. Проверим числовые признаки на утечку (урок «Категориальные признаки и утечки»). Возьмите столбцы

`tenure_months`, `app`, `visits_90d`, `days_since_last`, `avg_check`, `delivery_share`, `promo_used`, `complaints`, `card_blocked`

и разделите их и `churned`: `train_test_split(..., test_size=0.25, random_state=42, stratify=g["churned"])`. Обучите `DecisionTreeClassifier(max_depth=3, random_state=0)` и сохраните:

- `acc_all` — долю правильных на тесте (3 знака);
- `top_feature` — признак с наибольшей важностью (`feature_importances_`);
- `acc_clean` — то же дерево без этого признака (3 знака);
- `blocked_churn` — долю ушедших среди гостей с `card_blocked == 1` (3 знака).

*Подсказки (2): `dsc.hint('ch06/09-project/leak')`*

In [ ]:
%%task ch06/09-project/leak
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
cols = ["tenure_months", "app", "visits_90d", "days_since_last", "avg_check",
        "delivery_share", "promo_used", "complaints", "card_blocked"]

### Шаг 4. Признаки и разбиение · база

`g` с заполненным `avg_check` уже готова. Соберите:

- `y` — столбец `churned`;
- `X` — `pd.get_dummies(..., dtype=int)` от всех столбцов `g`, кроме `guest_id`, `churned` и `card_blocked`;
- `X_train`, `X_test`, `y_train`, `y_test` — `train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)`.

Сохраните `train_rate` и `test_rate` — доли ушедших в обучающей и тестовой частях (3 знака).

*Подсказки (2): `dsc.hint('ch06/09-project/prepare')`*

In [ ]:
%%task ch06/09-project/prepare
import pandas as pd
from sklearn.model_selection import train_test_split

### Шаг 5. Базовая модель и правило · база

Оцените на фолдах `kf` по **обучающей** части (`X_train`, `y_train`) две модели без обучения:

- «никто не уйдёт» — `base_cv`;
- правило заказчика «уйдёт, если не был больше 45 дней» (`days_since_last > 45`) — `rule_cv`.

Обе — средняя по фолдам доля правильных ответов (3 знака).

*Подсказки (2): `dsc.hint('ch06/09-project/baseline')`*

In [ ]:
%%task ch06/09-project/baseline
import numpy as np

### Шаг 6. Глубина дерева · база

Подберите глубину дерева кросс-валидацией на обучающей части: для глубин от 1 до 10 посчитайте среднюю долю правильных `DecisionTreeClassifier(max_depth=..., random_state=0)` по фолдам `kf`. Сохраните словарь `cv_depth` (глубина → среднее), `best_depth` (при равенстве — меньшая) и `best_cv` — среднее для неё (3 знака).

*Подсказки (2): `dsc.hint('ch06/09-project/depth')`*

In [ ]:
%%task ch06/09-project/depth
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
cv_depth = {}

### Шаг 7. Лес и логистическая регрессия · посложнее

Курс сохранил лучшую глубину из шага 6 в `best_depth`. Сравните на фолдах `kf` по обучающей части три модели:

- `"дерево"` — `DecisionTreeClassifier(max_depth=best_depth, random_state=0)`;
- `"лес"` — `RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=0)`;
- `"логрегрессия"` — `LogisticRegression(max_iter=2000)`.

Сохраните таблицу `table` (индекс — названия, столбцы `mean` и `std`, 3 знака, по убыванию `mean`) и `winner` — название лучшей модели.

*Подсказки (2): `dsc.hint('ch06/09-project/models')`*

In [ ]:
%%task ch06/09-project/models
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

### Шаг 8. Финальная проверка на тесте · база

Победитель шага 7 — лес (его отрыв от дерева меньше разброса между фолдами; берём лес за устойчивость — см. разбор шага 7). Обучите `RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=0)` на всей обучающей части и сохраните в `model`. Теперь — один раз — посчитайте на тесте (3 знака):

- `test_acc` — долю правильных ответов леса;
- `base_test` — модели «никто не уйдёт»;
- `rule_test` — правила `days_since_last > 45`.

*Подсказки (2): `dsc.hint('ch06/09-project/final')`*

In [ ]:
%%task ch06/09-project/final
from sklearn.ensemble import RandomForestClassifier

### Шаг 9. Список для акции · посложнее

Курс обучил лес из шага 8 — он в `model`. Бюджета хватает на 10 % гостей. Отберите их по риску на тестовой части:

- `risk` — Series с вероятностью ухода (`predict_proba(X_test)[:, 1]`) и индексом `X_test`;
- `top` — 10 % тестовых гостей с наибольшим риском: `risk.nlargest(60)`;
- `top_ids` — список их `guest_id` (из `g`) в том же порядке;
- `top_rate` — доля ушедших среди них (3 знака);
- `lift` — во сколько раз `top_rate` больше доли ушедших во всём тесте (1 знак);
- `already_gone` — доля гостей в `top`, которые не были больше 45 дней, то есть уже известны по правилу (3 знака).

*Подсказки (2): `dsc.hint('ch06/09-project/top')`*

In [ ]:
%%task ch06/09-project/top
import pandas as pd

### Шаг 10. Тихий риск · со звёздочкой

Акция нужнее тем, кто **ещё ходит** — был не позже 45 дней назад: правило про них молчит. Возьмите таких гостей тестовой части (`days_since_last <= 45`) и сохраните:

- `quiet_rate` — долю ушедших среди них (3 знака);
- `k` — 10 % от их числа, `int(len(...) * 0.1)`;
- `model_rate` — долю ушедших среди `k` гостей с наибольшим риском по модели (`nlargest(k)`, 3 знака);
- `days_rate` — то же, если отобрать `k` гостей с наибольшим `days_since_last` (тоже `nlargest(k)`, 3 знака) — «правило, доведённое до предела».

Лес из шага 8 — в `model`.

*Подсказки (2): `dsc.hint('quiet')` · задача по желанию*

In [ ]:
%%task ch06/09-project/quiet
import pandas as pd

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()